# Tai Dinh, Week 3, Data Preprocessing

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# Loading data
Loading 13 months of data and combining into one dataframe

In [ ]:
june25 = pd.read_csv('/content/CRMLSSold202506.csv')
july25 = pd.read_csv('/content/CRMLSSold202507.csv')
aug25 = pd.read_csv('/content/CRMLSSold202508.csv')
sept25 = pd.read_csv('/content/CRMLSSold202509.csv')
oct25 = pd.read_csv('/content/CRMLSSold202510.csv')
nov25 = pd.read_csv('/content/CRMLSSold202511.csv')
dec25 = pd.read_csv('/content/CRMLSSold202512.csv')
jan26 = pd.read_csv('/content/CRMLSSold202601.csv')
feb26 = pd.read_csv('/content/CRMLSSold202602.csv')
mar26 = pd.read_csv('/content/CRMLSSold202603.csv')
april26 = pd.read_csv('/content/CRMLSSold202604.csv')
may26 = pd.read_csv('/content/CRMLSSold202605.csv')
june26 = pd.read_csv('/content/CRMLSSold202606.csv')

/tmp/ipykernel_451/1754224898.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  june25 = pd.read_csv('/content/CRMLSSold202506.csv')
/tmp/ipykernel_451/1754224898.py:8: DtypeWarning: Columns (4,74) have mixed types. Specify dtype option on import or set low_memory=False.
  jan26 = pd.read_csv('/content/CRMLSSold202601.csv')


In [ ]:
dfs = [june25, july25, aug25, sept25, oct25, nov25, dec25, jan26, feb26, mar26, april26, may26, june26]
df = pd.concat(dfs, ignore_index=True)
df.shape

(283176, 78)

In [ ]:
df = df.drop(columns = ["BuyerAgentAOR", "ListAgentAOR"])
df.head()

,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,...,LotSizeDimensions,LotSizeArea,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict
0,NaN,False,NaN,NaN,False,5000.0,542181398,ops@downtowncondoguys.com,2025-06-26,3995.0,...,NaN,NaN,NaN,NaN,1.0,NaN,92101,0.0,NaN,NaN
1,NaN,True,NaN,NaN,NaN,110000.0,540760713,nowjoanne@gmail.com,2025-06-13,130000.0,...,NaN,355936.0,NaN,False,NaN,NaN,93550,0.0,355936.0,NaN
2,NaN,True,NaN,NaN,NaN,1545000.0,525608302,Austin_Brown@pacificplayarealty.com,2025-06-23,1581750.0,...,88x231,20212.0,NaN,False,NaN,NaN,90008,NaN,20212.0,NaN
3,NaN,True,NaN,NaN,False,889000.0,523319952,hutton@cbappteam.com,2025-06-13,890000.0,...,NaN,9600.0,0.0,True,2.0,Rim of the World,92352,0.0,9600.0,NaN
4,Laminate,True,NaN,NaN,False,1700.0,518730969,mannybehar@yahoo.com,2025-06-01,2100.0,...,NaN,NaN,NaN,NaN,0.0,NaN,92126,0.0,NaN,NaN


In [ ]:
#focus on residential single family homes
df = df[(df["PropertyType"] == "Residential") & (df["PropertySubType"] == "SingleFamilyResidence")].copy()

In [ ]:
#shape and info on filtered dataset
print(df.shape)
df.info()

(143078, 76)
<class 'pandas.core.frame.DataFrame'>
Index: 143078 entries, 3 to 283173
Data columns (total 76 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   Flooring                      92226 non-null   object 
 1   ViewYN                        130030 non-null  object 
 2   WaterfrontYN                  74 non-null      object 
 3   BasementYN                    3464 non-null    object 
 4   PoolPrivateYN                 131892 non-null  object 
 5   OriginalListPrice             142774 non-null  float64
 6   ListingKey                    143078 non-null  int64  
 7   ListAgentEmail                142697 non-null  object 
 8   CloseDate                     143078 non-null  object 
 9   ClosePrice                    143078 non-null  float64
 10  ListAgentFirstName            142120 non-null  object 
 11  ListAgentLastName             143067 non-null  object 
 12  Latitude                      143065

In [ ]:
df.describe()

,OriginalListPrice,ListingKey,ClosePrice,Latitude,Longitude,LivingArea,ListPrice,DaysOnMarket,FireplacesTotal,AboveGradeFinishedArea,...,ElementarySchoolDistrict,BelowGradeFinishedArea,CoveredSpaces,Stories,LotSizeArea,MainLevelBedrooms,GarageSpaces,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict
count,1.427740e+05,1.430780e+05,1.430780e+05,143065.000000,143065.000000,143004.000000,1.430780e+05,143078.000000,0.0,0.0,...,0.0,1021.000000,0.0,128148.000000,1.406420e+05,86970.000000,137448.000000,101374.000000,1.406290e+05,0.0
mean,1.383857e+06,1.134512e+09,1.342062e+06,34.735839,-118.600452,2051.894708,1.269288e+06,39.862900,NaN,NaN,...,NaN,66.991185,NaN,1.352670,2.098009e+04,2.265229,2.007274,107.523592,3.756355e+05,NaN
std,8.063208e+06,1.906691e+07,7.893845e+06,1.768924,3.233273,1049.593977,1.589906e+06,53.140488,NaN,NaN,...,NaN,292.655253,NaN,0.477803,1.128845e+06,1.456901,3.293176,359.199367,1.751203e+07,NaN
min,0.000000e+00,4.217759e+08,0.000000e+00,-22.863239,-124.193201,0.000000,8.000000e+03,-265.000000,NaN,NaN,...,NaN,0.000000,NaN,1.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,NaN
25%,6.370000e+05,1.117480e+09,6.250000e+05,33.761153,-119.161909,1388.000000,6.250000e+05,8.000000,NaN,NaN,...,NaN,0.000000,NaN,1.000000,5.403000e+03,1.000000,2.000000,0.000000,5.663000e+03,NaN
50%,8.990000e+05,1.136709e+09,8.950000e+05,34.084240,-118.031509,1823.000000,8.950000e+05,20.000000,NaN,NaN,...,NaN,0.000000,NaN,1.000000,7.095000e+03,3.000000,2.000000,0.000000,7.285000e+03,NaN
75%,1.449000e+06,1.151934e+09,1.425000e+06,34.848756,-117.260040,2444.000000,1.399999e+06,52.000000,NaN,NaN,...,NaN,0.000000,NaN,2.000000,9.995000e+03,3.000000,2.000000,132.000000,1.045400e+04,NaN
max,1.302000e+09,1.176283e+09,9.895000e+08,81.000000,120.432670,56500.000000,1.375000e+08,2177.000000,NaN,NaN,...,NaN,3490.000000,NaN,2.000000,4.187423e+08,44.000000,600.000000,23750.000000,1.938943e+09,NaN


# Handling Missing or Incorrect Values

In [ ]:
#drop properties with 0 living area, not valid data
df = df[df["LivingArea"] > 0].reset_index(drop=True)
df.shape

#impute missing numeric values in specified cols with median(resistant to high outliers in dataset)
med_imputer = SimpleImputer(strategy='median')

#impute certain numeric columns with 0/1 if not there
zero_imputer = SimpleImputer(strategy='constant', fill_value=0)
one_imputer = SimpleImputer(strategy = "constant", fill_value = 1)

#impute boolean columns with False if not there
bool_imputer = SimpleImputer(strategy='constant', fill_value=False)

#flag categorical variables with missing if not there
cat_imputer = SimpleImputer(strategy='constant', fill_value='None')

num_cols = ["LivingArea", "YearBuilt"]
zero_cols = ["ParkingTotal", "BathroomsTotalInteger", "GarageSpaces", "AssociationFee"]
bool_cols = ["AttachedGarageYN", "PoolPrivateYN", "ViewYN"]
cat_cols = ["AssociationFeeFrequency"]

df["ParkingTotal"] = np.abs(df["ParkingTotal"])

for col in num_cols:
    df[col] = med_imputer.fit_transform(df[[col]])

for col in zero_cols:
    df[col] = zero_imputer.fit_transform(df[[col]])

for col in bool_cols:
    df[col] = bool_imputer.fit_transform(df[[col]])[:, 0]

#default is one story if not specified
df["Stories"] = one_imputer.fit_transform(df[["Stories"]])

for col in cat_cols:
    df[col] = cat_imputer.fit_transform(df[[col]])[:, 0]

# Adding School District Mapping

In [ ]:
geo_df = gpd.read_file("/content/DistrictAreas2526_-284845464123469011.geojson")

In [ ]:
geo_df = geo_df[geo_df["DistrictType"] == "Unified"]
geo_df.head()

,OBJECTID,Year,FedID,CDCode,CDSCode,CountyName,DistrictName,DistrictType,GradeLow,GradeHigh,...,MIGcount,MIGpct,SWDcount,SWDpct,SEDcount,SEDpct,DistrctAreaSqMi,LocaleCode,LocaleDesc,geometry
0,1,2025-26,0601770,0161119,01611190000000,Alameda,Alameda Unified,Unified,PK,12,...,0,0.0,1302,12.1,4259,39.5,11.248886,21,"21 - Suburban, Large","MULTIPOLYGON (((-13606222.82 4540862.699, -136..."
1,2,2025-26,0601860,0161127,01611270000000,Alameda,Albany City Unified,Unified,PK,12,...,0,0.0,363,9.7,1247,33.3,1.789975,21,"21 - Suburban, Large","POLYGON ((-13612893.866 4565099.707, -13612896..."
2,3,2025-26,0604740,0161143,01611430000000,Alameda,Berkeley Unified,Unified,PK,12,...,0,0.0,1118,11.9,2710,28.8,10.434281,12,"12 - City, Midsize","POLYGON ((-13609482.48 4565074.597, -13609483...."
3,4,2025-26,0607800,0161150,01611500000000,Alameda,Castro Valley Unified,Unified,PK,12,...,2,0.0,1186,12.2,3784,39.0,66.885261,21,"21 - Suburban, Large","MULTIPOLYGON (((-13582508.535 4529067.071, -13..."
4,5,2025-26,0612630,0161168,01611680000000,Alameda,Emery Unified,Unified,PK,12,...,0,0.0,98,16.1,407,66.7,1.273923,21,"21 - Suburban, Large","POLYGON ((-13613999.038 4555592.769, -13614126..."


In [ ]:
df_point = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['Longitude'], df['Latitude']),
    crs="EPSG:4326")
df_point.head()

,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,...,LotSizeArea,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,geometry
0,NaN,True,NaN,NaN,False,889000.0,523319952,hutton@cbappteam.com,2025-06-13,890000.0,...,9600.0,0.0,True,2.0,Rim of the World,92352,0.0,9600.0,NaN,POINT (-117.22104 34.26469)
1,Wood,True,NaN,NaN,False,1899999.0,1118606385,chase.campen@compass.com,2025-06-30,1876384.0,...,10400.0,NaN,False,0.0,NaN,90046,0.0,10400.0,NaN,POINT (-118.39032 34.10798)
2,NaN,False,NaN,NaN,False,NaN,1118606192,stanleylo@greenbanker.com,2025-06-30,4820000.0,...,22505.0,NaN,False,3.0,Other,94010,0.0,22505.0,NaN,POINT (-122.38823 37.56743)
3,NaN,True,NaN,NaN,False,865000.0,1118604114,matt@majorleaguesocal.com,2025-06-30,865000.0,...,4800.0,3.0,False,2.0,Placentia-Yorba Linda Unified,92886,0.0,4800.0,NaN,POINT (-117.77778 33.90606)
4,"Carpet,Laminate",False,NaN,NaN,False,875000.0,1118603794,brianrowland.homes@gmail.com,2025-06-30,875000.0,...,5500.0,NaN,False,4.0,NaN,94546,0.0,5500.0,NaN,POINT (-122.05942 37.70592)


In [ ]:
#combine original dataset with geographic school district data
if geo_df.crs != df_point.crs:
    df_point = df_point.to_crs(geo_df.crs)

combined_df = gpd.sjoin(df_point, geo_df, how="left")
combined_df.head()

,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,...,HOMpct,MIGcount,MIGpct,SWDcount,SWDpct,SEDcount,SEDpct,DistrctAreaSqMi,LocaleCode,LocaleDesc
0,NaN,True,NaN,NaN,False,889000.0,523319952,hutton@cbappteam.com,2025-06-13,890000.0,...,1.3,0.0,0.0,447.0,15.5,1778.0,61.5,113.055142,31,"31 - Town, Fringe"
1,Wood,True,NaN,NaN,False,1899999.0,1118606385,chase.campen@compass.com,2025-06-30,1876384.0,...,2.4,1294.0,0.3,81461.0,16.4,410978.0,82.7,699.955103,11,"11 - City, Large"
2,NaN,False,NaN,NaN,False,NaN,1118606192,stanleylo@greenbanker.com,2025-06-30,4820000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,True,NaN,NaN,False,865000.0,1118604114,matt@majorleaguesocal.com,2025-06-30,865000.0,...,13.9,15.0,0.1,3161.0,14.2,10492.0,47.1,39.259300,21,"21 - Suburban, Large"
4,"Carpet,Laminate",False,NaN,NaN,False,875000.0,1118603794,brianrowland.homes@gmail.com,2025-06-30,875000.0,...,0.9,2.0,0.0,1186.0,12.2,3784.0,39.0,66.885261,21,"21 - Suburban, Large"


In [ ]:
#impute missing values with "none" if there is no matching school district
combined_df["DistrictName"] = cat_imputer.fit_transform(combined_df[["DistrictName"]])[:, 0]

In [ ]:
district_stats = combined_df["DistrictName"].value_counts()
combined_df["DistrictName"] = combined_df["DistrictName"].apply(lambda x: x if district_stats[x] >= 650 else "Other")
combined_df.value_counts("DistrictName")

,count
DistrictName,
Other,36912
None,34605
Los Angeles Unified,14206
San Diego Unified,3864
Capistrano Unified,2639
Desert Sands Unified,2607
Palm Springs Unified,2324
Oakland Unified,1860
Corona-Norco Unified,1776


# Encoding Variables

In [ ]:
#encodes categorical variables
ohe_encoder = OneHotEncoder(drop = 'first', sparse_output=False)
cols_to_encode = ["AssociationFeeFrequency", "CountyOrParish", "StateOrProvince", "DistrictName"]

transformer = ColumnTransformer(transformers=[("cat_encoder", ohe_encoder, cols_to_encode)],remainder="passthrough", verbose_feature_names_out=False)
transformer.set_output(transform="pandas")
combined_df = transformer.fit_transform(combined_df)

In [ ]:
#adding column that checks if property has nearby school
df["has_elementary_school"] = df["ElementarySchool"].notna().astype(int)

# Additional Feature Engineering

In [ ]:
combined_df["Bed/Bath Ratio"] = combined_df["BedroomsTotal"] / combined_df["BathroomsTotalInteger"]
combined_df["Living Area per Bedroom"] = combined_df["LivingArea"] / combined_df["BedroomsTotal"]
combined_df["Age"] = 2026 - combined_df["YearBuilt"]

In [ ]:
#handle outliers
combined_df["Bed/Bath Ratio"] = combined_df["Bed/Bath Ratio"].replace([np.inf, -np.inf], 0)
combined_df["Living Area per Bedroom"] = combined_df["Bed/Bath Ratio"].replace([np.inf, -np.inf], 0)
combined_df["Age"] = combined_df["Age"].fillna(0)

In [ ]:
combined_df["Bed/Bath Ratio"] = med_imputer.fit_transform(combined_df[["Bed/Bath Ratio"]])
combined_df["Living Area per Bedroom"] = med_imputer.fit_transform(combined_df[["Living Area per Bedroom"]])

In [ ]:
combined_df["Age"].isna().sum()

np.int64(0)

# Train/Test Split

In [ ]:
combined_df['CloseDate'] = pd.to_datetime(combined_df['CloseDate'])
combined_df['CloseMonth'] = combined_df['CloseDate'].dt.month
combined_df['CloseYear'] = combined_df['CloseDate'].dt.year

The latest month available in the dataset is June 2026. This will be the test set. The validation set is May 2026 and the rest of the months are the training set.

In [ ]:
test_month = 6
test_year = 2026

valid_month = 5
valid_year = 2026

train_df = combined_df[(combined_df['CloseMonth'] < valid_month) | (combined_df['CloseYear'] < valid_year)].copy()
valid_df = combined_df[(combined_df['CloseMonth'] == valid_month) & (combined_df['CloseYear'] == valid_year)].copy()
test_df = combined_df[(combined_df['CloseMonth'] == test_month) & (combined_df['CloseYear'] == test_year)].copy()

In [ ]:
train_df = train_df[(train_df["ClosePrice"] <= 600000000) & (train_df["ClosePrice"] > 0)]
valid_df = valid_df[(valid_df["ClosePrice"] <= 600000000) & (valid_df["ClosePrice"] > 0)]

In [ ]:
print(f"Shape of training set: {train_df.shape}")
print(f"Shape of validation set: {valid_df.shape}")
print(f"Shape of test set: {test_df.shape}")

Shape of training set: (118076, 245)
Shape of validation set: (12015, 245)
Shape of test set: (12845, 245)


# Exporting Data

In [ ]:
#export As CSV
train_df.to_csv('train_df.csv', index=False)
valid_df.to_csv('valid_df.csv',index=False)
test_df.to_csv('test_df.csv',index=False)

files.download('train_df.csv')
files.download('valid_df.csv')
files.download('test_df.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>